In [1]:
import os
from datasets import load_dataset
from langchain.text_splitter import TokenTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain.vectorstores import FAISS
from langchain.chains import RetrievalQA
from langchain.schema import Document

c:\Users\karan\PycharmProjects\ragvsfinetuning\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
dataset = load_dataset("prsdm/Machine-Learning-QA-dataset", split="train")
raw_texts = [q + "\n\nAnswer: " + a for q, a in zip(dataset["Question"], dataset["Answer"])]


In [9]:
splitter = TokenTextSplitter(chunk_size=300, chunk_overlap=50)
docs = []
for idx, text in enumerate(raw_texts):
    for i, chunk in enumerate(splitter.split_text(text)):
        doc =  Document(
            page_content=chunk,
            metadata={"source_id": idx, "chunk_index": i}
        )
        docs.append(doc)
print(f"Created {len(docs)} chunks.")

Created 101 chunks.


In [ ]:
embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

In [11]:
index_path = 'embeddings_db'
vector_store = FAISS.from_documents(docs, embeddings)
vector_store.save_local(index_path)

In [13]:
vector_store = FAISS.load_local(index_path, embeddings , allow_dangerous_deserialization= True)


In [ ]:
chat_model = ChatGoogleGenerativeAI(model_name="gemini-flash-2.0")
rag = RetrievalQA.from_chain_type(
        llm=chat_model,
        chain_type="stuff",
        retriever=vector_store.as_retriever(search_kwargs={"k": 5}),
        return_source_documents=True,
    )
query = ""
result = rag(query)
print("\nAnswer:\n", result["result"])